# 05 - Local completion heads (decision gate G3)
Three variants: a pointwise head fitted post hoc on the frozen backbone, the dependency-propagating
head (reads its neighbours' previous done probabilities, like the automaton of Proposition 3), and
both trained jointly as an auxiliary task. Target: position i is done at step t if it is correct
at t and at every later step. Halting: every position reports done with probability > tau.

**G3 (pre-registered):** on OOD-depth the best completion-head variant beats the best baseline by
>= 10 accuracy points or >= 50% fewer premature stops, CI excluding zero across seeds.

In [ ]:
# parameters
CONFIGS = ["configs/main_addition.yaml", "configs/main_parity.yaml", "configs/main_gates.yaml"]
TASKS = ["addition", "parity"]
SPLITS = ["id_test", "ood_depth", "ood_length", "ood_both"]

In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [ ]:
from glwm2.eval.results import configs_from_yaml, finished, curve_rows, step_acc
from glwm2.eval.analysis import (ENTRIES, BASELINES, OURS, POOLED_JOINT, CONF_RULES, entry_rows, entry_instances,
                                 summary_table, g2_check, g3_check, binned, premature_ci, length_leakage_ci)
from glwm2.eval.plotting import savefig, FAMILY_COLORS
from glwm2.utils.io import write_json
cfgs = {t: [c for f in CONFIGS for c in configs_from_yaml(PROJECT_ROOT / f) if c.task == t] for t in TASKS}
for t in TASKS:
    print(t, f"{len(finished(cfgs[t]))}/{len(cfgs[t])} runs finished, seeds:",
          sorted({c.seed for c in finished(cfgs[t])}))
rows = {t: entry_rows(cfgs[t]) for t in TASKS}

In [ ]:
g3 = {}
for t in TASKS:
    for split in ["ood_depth", "ood_both"]:
        inst = entry_instances(cfgs[t], split)
        r = g3_check(inst)
        g3[f"{t}/{split}"] = r
        print(t, split, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in r.items() if k not in ("acc_diff", "prem_diff")})
        if r:
            print("   acc diff (point, lo, hi):", np.round(r["acc_diff"], 3), "  premature diff:", np.round(r["prem_diff"], 3))
G3 = all(g3[f"{t}/ood_depth"].get("G3", False) for t in TASKS)
print("G3 on OOD-depth (pre-registered):", G3)

In [ ]:
for t in TASKS:
    keep = OURS + ["oracle_stable", "max_softmax", "entropy", "stability", "trm_post", "act", "ponder", "trm_joint"]
    sub = rows[t][rows[t].entry.isin(keep)]
    print(f"== {t}"); display(sub.pivot_table(index="entry", columns="split", values=["acc", "premature", "mean_steps"], aggfunc="mean")
                              .reindex([e for e in ENTRIES if e in keep]).round(3))

In [ ]:
# completion-head calibration (ECE of per-position done probabilities against done targets)
from glwm2.train.trainer import run_dirs
from glwm2.utils.io import read_json
cal = []
for t in TASKS:
    for c in finished(cfgs[t]):
        if c.method not in ("base", "done_joint", "done_prop_joint"): continue
        for split, d in read_json(run_dirs(c)[1] / "calibration.json").items():
            for head, ece in d.items():
                cal.append(dict(task=t, method=c.method, head=head, split=split, seed=c.seed, ece=ece))
cal = pd.DataFrame(cal)
display(cal.pivot_table(index=["task", "method", "head"], columns="split", values="ece", aggfunc="mean").round(4))

In [ ]:
# H4: does joint training distort the per-step accuracy curve? AUC of ID-val accuracy over steps 1..T_max
auc = []
for t in TASKS:
    for c in finished(cfgs[t]):
        a = step_acc(c)["id_val"]
        auc.append(dict(task=t, method=c.method, seed=c.seed, auc=float(np.mean(a))))
auc = pd.DataFrame(auc)
h4 = auc.pivot_table(index="method", columns="task", values="auc", aggfunc="mean")
h4_diff = (h4 - h4.loc["base"]) * 100
display(h4.round(4)); print("difference from base, accuracy points:"); display(h4_diff.round(2))

In [ ]:
# how the done signal spreads over positions and steps (deep OOD-both addition instance)
from glwm2.eval.results import per_instance
from glwm2.data.splits import load_split
from glwm2.eval.trajectory import per_position_rho
c = next(c for c in finished(cfgs["addition"]) if c.method == "base")
pi = per_instance(c)
data = load_split("addition", "ood_both")
heads = [h for h in ["done_post", "done_prop_post"] if h in pi["pos_done"]["ood_both"]]
k = int(torch.argmax(data["depth"][:64]))
L = int(data["length"][k]); rho = per_position_rho("addition", {kk: v[k:k+1] for kk, v in data.items()})[0, :L+1].numpy()
fig, axes = plt.subplots(1, len(heads), figsize=(7.0, 2.4), squeeze=False)
for ax, h in zip(axes[0], heads):
    P = pi["pos_done"]["ood_both"][h][:, k, :L+1].float().numpy()
    im = ax.imshow(P.T, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1, extent=[0.5, P.shape[0] + 0.5, -0.5, L + 0.5])
    ax.plot(rho, np.arange(L + 1), color="#D55E00", lw=1)
    ax.set_title(f"{h}: done probability"); ax.set_xlabel("step t"); ax.set_ylabel("position i")
fig.colorbar(im, ax=axes[0].tolist(), shrink=0.8)
WRITTEN.append(savefig(fig, "figA_done_spread"))

In [ ]:
WRITTEN.append(write_json(RESULTS_DIR / "gates" / "g3.json", {"G3": G3, "detail": g3, "h4_auc_diff_points": h4_diff.to_dict()}))

In [ ]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))